# Chemical Phase Interface Assessment — Solution

Seed-free **leave-experiment-out** image-regression for an ordinal `interface_burden` (0–100),
scored ~80% on the ordinal burden ZONE. Pipeline (see `Approach.md`):
- **Honest CV** by inferred experiment groups (exact image size ≈ experiment id; used ONLY for folds).
- **ConvNeXt-V2 (nano + femto) + GeM**, dual **SORD ordinal-PMF + BCE-regression** heads.
- **Bayes-optimal expected-cost decision** over the exact metric (+ temperature calibration).
- **Multi-seed ensemble** (per-fold variance is large) + hflip TTA. Aggressive color/style aug HURTS
  here (color/intensity is the genuine turbidity signal), so augmentation is deliberately light.

Reads `./dataset/public/`, writes `./working/submission.csv`. Set `FAST=1` for a quick smoke.


In [ ]:
"""Exact challenge metric for Chemical Phase Interface Assessment + ordinal helpers.

The official metric (LOWER is better) is dominated (0.80 weight) by ordinal zone
accuracy. This module provides:
  - evaluate(): byte-for-byte copy of the official grader.
  - to_zone(): map burden -> ordinal zone {0,1,2,3}.
  - evaluate_components(): per-term breakdown for diagnostics.
  - best_constant(): the optimal single-value baseline on a label array.
  - decision_value_from_zone_probs(): Bayes-optimal output scalar given P(zone)
    and a regression point estimate, minimizing expected metric. This is the
    key exploit of a known, piecewise-constant metric.
"""
import numpy as np

SEVERITY_BINS = np.array([0.0, 12.0, 35.0, 48.0, 100.000001])
N_ZONES = 4
# Representative "safe center" of each zone (interior, away from boundaries),
# used as a default within-zone output when minimizing expected zone cost.
ZONE_CENTERS = np.array([6.0, 23.0, 41.0, 65.0])


def evaluate(y_true, y_pred):
    """Official grader. Lower is better. Returns float in [0, 100]."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0, 100)
    severity_bins = np.array([0.0, 12.0, 35.0, 48.0, 100.000001])
    absolute_gap = np.abs(y_pred - y_true)
    high_burden = y_true >= 48.0
    absolute_component = absolute_gap.mean()
    high_component = absolute_gap[high_burden].mean() if high_burden.any() else absolute_component
    true_zone = np.digitize(y_true, severity_bins) - 1
    pred_zone = np.digitize(y_pred, severity_bins) - 1
    zone_distance = np.abs(true_zone - pred_zone)
    zone_penalty = np.where(zone_distance == 0, 0.0, np.where(zone_distance == 1, 60.0, 100.0))
    zone_component = zone_penalty.mean()
    extreme_miss = ((y_true <= 12.0) & (y_pred > 25.0)) | ((y_true >= 48.0) & (y_pred < 40.0))
    extreme_component = extreme_miss.mean() * 100.0
    score = (
        0.05 * absolute_component
        + 0.05 * high_component
        + 0.80 * zone_component
        + 0.10 * extreme_component
    )
    return float(np.clip(score, 0.0, 100.0))


def to_zone(y):
    """Map burden value(s) to ordinal zone index {0,1,2,3}."""
    return np.digitize(np.asarray(y, dtype=float), SEVERITY_BINS) - 1


def evaluate_components(y_true, y_pred):
    """Return the 4 weighted components plus the total, for diagnostics."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, 100)
    absolute_gap = np.abs(y_pred - y_true)
    high = y_true >= 48.0
    abs_c = absolute_gap.mean()
    high_c = absolute_gap[high].mean() if high.any() else abs_c
    tz, pz = to_zone(y_true), to_zone(y_pred)
    zd = np.abs(tz - pz)
    zp = np.where(zd == 0, 0.0, np.where(zd == 1, 60.0, 100.0))
    zone_c = zp.mean()
    extreme = ((y_true <= 12.0) & (y_pred > 25.0)) | ((y_true >= 48.0) & (y_pred < 40.0))
    ext_c = extreme.mean() * 100.0
    total = float(np.clip(0.05 * abs_c + 0.05 * high_c + 0.80 * zone_c + 0.10 * ext_c, 0, 100))
    return {
        "total": total,
        "w_absolute": 0.05 * abs_c,
        "w_high": 0.05 * high_c,
        "w_zone": 0.80 * zone_c,
        "w_extreme": 0.10 * ext_c,
        "raw_mae": abs_c,
        "raw_mae_high": high_c,
        "raw_zone_penalty": zone_c,
        "zone_accuracy": float((zd == 0).mean()),
        "extreme_miss_rate": float(extreme.mean()),
    }


def best_constant(y_true, grid=None):
    """Find the single constant prediction minimizing the metric on y_true."""
    if grid is None:
        grid = np.arange(0, 100.01, 0.25)
    y_true = np.asarray(y_true, dtype=float)
    scores = [evaluate(y_true, np.full_like(y_true, c)) for c in grid]
    i = int(np.argmin(scores))
    return float(grid[i]), float(scores[i])


def _zone_cost_matrix():
    """cost[pred_zone, true_zone] from the 0/60/100 zone-distance penalty."""
    z = np.arange(N_ZONES)
    d = np.abs(z[:, None] - z[None, :])
    return np.where(d == 0, 0.0, np.where(d == 1, 60.0, 100.0))


ZONE_COST = _zone_cost_matrix()  # shape (pred, true)


def decision_value_from_zone_probs(zone_probs, reg_pred=None, true_prevalence=None):
    """Bayes-optimal output scalar(s) minimizing expected metric, given per-zone
    probabilities. zone_probs: (N,4). reg_pred: optional (N,) regression estimate
    used to pick a sensible within-zone value (and to feed the small MAE terms).

    Strategy: zone term dominates (0.80), so choose the output ZONE that minimizes
    expected zone cost E_true[cost[pred_zone, true_zone]]. Then emit a within-zone
    scalar: clip reg_pred into the chosen zone if available, else the zone center.
    """
    zone_probs = np.asarray(zone_probs, dtype=float)
    zone_probs = zone_probs / np.clip(zone_probs.sum(axis=1, keepdims=True), 1e-9, None)
    # expected zone cost for each candidate output zone: (N,4) = probs @ cost^T
    exp_cost = zone_probs @ ZONE_COST.T  # (N, pred_zone)
    chosen = np.argmin(exp_cost, axis=1)  # (N,)
    out = ZONE_CENTERS[chosen].astype(float)
    if reg_pred is not None:
        reg_pred = np.asarray(reg_pred, dtype=float)
        lo = SEVERITY_BINS[chosen]
        hi = SEVERITY_BINS[chosen + 1]
        # keep a small interior margin so boundary jitter doesn't flip zones
        margin = np.minimum(1.0, (hi - lo) * 0.15)
        out = np.clip(reg_pred, lo + margin, hi - margin)
    return out, chosen


def expected_cost_decision(pmf, centers, high_frac=0.384, out_grid=None, return_loss=False):
    """Bayes-optimal output minimizing the EXACT metric, integrating the model's
    predictive PMF over burden. pmf: (N,K) probabilities over `centers` (K,).

    Per-sample loss (the true metric factored by 1/N; argmin is unaffected):
        l(a,y) = 0.80*zone_pen + 0.05*|a-y| + (0.05/high_frac)*|a-y|*[y>=48] + 10*extreme
    where the high term's weight (0.05*N/N_high) reproduces the metric's
    conditional `high_component` mean. We grid-search a over out_grid.
    """
    pmf = np.asarray(pmf, dtype=float)
    centers = np.asarray(centers, dtype=float)
    if out_grid is None:
        out_grid = np.arange(0.0, max(centers.max(), 68.0) + 0.001, 0.5)
    A = np.asarray(out_grid, dtype=float)
    za = to_zone(A)[:, None]          # (nA,1)
    zc = to_zone(centers)[None, :]    # (1,K)
    dz = np.abs(za - zc)
    zone_pen = np.where(dz == 0, 0.0, np.where(dz == 1, 60.0, 100.0))
    mae = np.abs(A[:, None] - centers[None, :])
    high_w = 0.05 / max(high_frac, 1e-6)
    mae_high = mae * (centers[None, :] >= 48.0) * high_w
    extreme = (((centers[None, :] <= 12.0) & (A[:, None] > 25.0)) |
               ((centers[None, :] >= 48.0) & (A[:, None] < 40.0))).astype(float) * 10.0
    Lmat = 0.80 * zone_pen + 0.05 * mae + mae_high + extreme   # (nA, K)
    exp_loss = pmf @ Lmat.T            # (N, nA)
    best = exp_loss.argmin(1)
    out = A[best]
    if return_loss:
        return out, exp_loss[np.arange(len(out)), best]
    return out


def fit_temperature(pmf_logits, y_true, centers, grid=None):
    """Pick a single temperature T (scaling logits) that minimizes the OOF metric
    after the expected-cost decision. Returns (best_T, best_score)."""
    from scipy.special import softmax
    if grid is None:
        grid = np.concatenate([np.arange(0.3, 1.0, 0.1), np.arange(1.0, 4.01, 0.25)])
    best = (1.0, 1e9)
    for T in grid:
        pmf = softmax(np.asarray(pmf_logits) / T, axis=1)
        pred = expected_cost_decision(pmf, centers)
        s = evaluate(y_true, pred)
        if s < best[1]:
            best = (float(T), float(s))
    return best


In [ ]:
"""Chemical Phase Interface Assessment â€” core training/inference.

Design (grounded in EDA + 3-agent research, see notes.md / research_findings.md):
  * Honest CV: leave-experiment-out via size-based groups (exact size ~ experiment
    id; near-size@2px merge). Size is used ONLY for folds, never as a model input.
  * Model emits a DISTRIBUTION: ConvNeXt backbone + GeM pool -> SORD soft-ordinal
    head (K bins -> PMF) + auxiliary BCE-on-[0,1] regression head.
  * Decision = Bayes-optimal expected-cost over the EXACT metric (metric.py), on a
    temperature-calibrated OOF-averaged PMF. This exploits the known piecewise metric.
  * OOD augmentation: horizontal flip ONLY (vertical order is physical), color/hue
    jitter + light grayscale to kill palette shortcuts, blur/noise for glare, C-Mixup.

Run modes (env MODE): smoke | cv | predict. Paths via env DATA_ROOT / OUT_DIR.
"""
import os, sys, math, json, time, random
from dataclasses import dataclass, field, asdict
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

_STORE = None   # optional {id: uint8 (Hs,Ws,3)} RAM image store (set by run_cv/predict_test)

def load_store(split):
    """Load the pre-decoded image store from STORE_DIR into a {id: array} dict (views)."""
    sd = os.environ.get("STORE_DIR")
    if not sd or not (Path(sd) / f"{split}_imgs.npy").exists():
        return None
    imgs = np.load(Path(sd) / f"{split}_imgs.npy")
    ids = json.load(open(Path(sd) / f"{split}_ids.json"))
    return {i: imgs[k] for k, i in enumerate(ids)}


# ----------------------------- config -----------------------------
def _env(k, d, cast=str):
    v = os.environ.get(k)
    return cast(v) if v is not None else d

@dataclass
class Config:
    data_root: str = _env("DATA_ROOT", r"G:/ml/data/Chemical Phase dataset/public")
    out_dir: str = _env("OUT_DIR", "working")
    backbone: str = _env("BACKBONE", "convnextv2_nano.fcmae_ft_in22k_in1k")
    pretrained: bool = _env("PRETRAINED", 1, int) == 1
    img_h: int = _env("IMG_H", 384, int)
    img_w: int = _env("IMG_W", 224, int)
    n_folds: int = _env("N_FOLDS", 5, int)
    folds_to_run: str = _env("FOLDS", "", str)   # e.g. "0,1"; empty=all
    epochs: int = _env("EPOCHS", 18, int)
    batch_size: int = _env("BATCH", 16, int)
    lr: float = _env("LR", 2.5e-4, float)
    head_lr_mult: float = _env("HEAD_LR_MULT", 5.0, float)
    weight_decay: float = _env("WD", 0.05, float)
    warmup_frac: float = _env("WARMUP", 0.1, float)
    reg_weight: float = _env("REG_W", 0.3, float)
    n_bins: int = _env("NBINS", 69, int)
    bin_lo: float = _env("BIN_LO", 0.0, float)
    bin_hi: float = _env("BIN_HI", 68.0, float)
    sord_sigma: float = _env("SORD_SIGMA", 2.0, float)   # in burden units
    ema_decay: float = _env("EMA", 0.999, float)
    cmix_p: float = _env("CMIX_P", 0.5, float)
    cmix_alpha: float = _env("CMIX_ALPHA", 0.3, float)
    color_jitter: float = _env("CJ", 0.3, float)
    gray_p: float = _env("GRAY_P", 0.1, float)
    mixstyle: bool = _env("MIXSTYLE", 0, int) == 1
    mixstyle_p: float = _env("MIXSTYLE_P", 0.5, float)
    drop_path: float = _env("DROP_PATH", 0.0, float)
    num_workers: int = _env("NUM_WORKERS", 0, int)
    seed: int = _env("SEED", 42, int)
    fold_seed: int = _env("FOLD_SEED", 42, int)   # FIXED across seeds so OOF aligns for ensembling
    smoke: bool = _env("SMOKE", 0, int) == 1
    cache: bool = _env("CACHE", 0, int) == 1

    @property
    def centers(self):
        return np.linspace(self.bin_lo, self.bin_hi, self.n_bins)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ----------------------------- groups / folds -----------------------------
def image_sizes(df, data_root):
    ws, hs = [], []
    for p in df.image_path:
        with Image.open(Path(data_root) / p) as im:
            ws.append(im.size[0]); hs.append(im.size[1])
    return np.array(ws), np.array(hs)

def compute_groups(df, data_root, tol_abs=2):
    """Leave-experiment-out groups from image size only (reproducible, no model).
    Exact-size union + merge sizes within tol_abs px in BOTH dims. Matches the
    near-size@2px grouping validated in eda/group_infer.py (272 groups)."""
    W, H = image_sizes(df, data_root)
    n = len(df)
    parent = list(range(n))
    def find(a):
        while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    by = {}
    for i in range(n): by.setdefault((W[i], H[i]), []).append(i)
    for ids in by.values():
        for i in ids[1:]: union(ids[0], i)
    sizes = list(by.keys()); reps = {s: by[s][0] for s in sizes}
    S = np.array(sizes, dtype=float)
    for a in range(len(S)):
        for b in range(a + 1, len(S)):
            if abs(S[a, 0]-S[b, 0]) <= tol_abs and abs(S[a, 1]-S[b, 1]) <= tol_abs:
                union(reps[sizes[a]], reps[sizes[b]])
    lab = np.array([find(i) for i in range(n)])
    _, lab = np.unique(lab, return_inverse=True)
    return lab

def make_folds(df, groups, n_folds, seed):
    """StratifiedGroupKFold on (zone x group): balance zones, never split a group."""
    from sklearn.model_selection import StratifiedGroupKFold
    zone = to_zone(df.interface_burden.values)
    skf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold = np.full(len(df), -1)
    for f, (_, va) in enumerate(skf.split(df, zone, groups)):
        fold[va] = f
    assert (fold >= 0).all()
    return fold

# ----------------------------- targets -----------------------------
def soft_ordinal_targets(y, centers, sigma):
    """SORD: soft label = softmax(-(center-y)^2 / (2 sigma^2)) over bins."""
    d2 = (centers[None, :] - np.asarray(y)[:, None]) ** 2
    logits = -d2 / (2 * sigma ** 2)
    logits -= logits.max(1, keepdims=True)
    p = np.exp(logits); p /= p.sum(1, keepdims=True)
    return p.astype(np.float32)

# ----------------------------- model -----------------------------
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.ones(1) * p); self.eps = eps
    def forward(self, x):                      # (B,C,H,W)
        return x.clamp(min=self.eps).pow(self.p).mean((-2, -1)).pow(1.0 / self.p)

class MixStyle(nn.Module):
    """Mix instance-level feature statistics across the batch (Zhou et al. ICLR'21).
    Applied at EARLY stages only (late stages carry label info). Train-time only."""
    def __init__(self, p=0.5, alpha=0.1, eps=1e-6):
        super().__init__(); self.p = p; self.eps = eps
        self.beta = torch.distributions.Beta(alpha, alpha)
    def forward(self, x):
        if not self.training or random.random() > self.p or x.size(0) < 2:
            return x
        dt = x.dtype
        x = x.float()                                  # fp32 for stable instance stats (avoids fp16 NaN)
        mu = x.mean([2, 3], keepdim=True); var = x.var([2, 3], keepdim=True)
        sig = (var + self.eps).sqrt()
        xn = (x - mu) / sig
        lam = self.beta.sample((x.size(0), 1, 1, 1)).to(x.device).float()
        perm = torch.randperm(x.size(0), device=x.device)
        mu_mix = mu * lam + mu[perm] * (1 - lam)
        sig_mix = sig * lam + sig[perm] * (1 - lam)
        return (xn * sig_mix + mu_mix).to(dt)

class Net(nn.Module):
    def __init__(self, backbone, n_bins, pretrained=True, mixstyle=False, mixstyle_p=0.5, drop_path=0.0):
        super().__init__()
        import timm
        self.bb = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="",
                                    drop_path_rate=drop_path)
        C = self.bb.num_features
        self.pool = GeM()
        self.drop = nn.Dropout(0.1)
        self.cls = nn.Linear(C, n_bins)
        self.reg = nn.Linear(C, 1)
        self.ms = MixStyle(p=mixstyle_p) if mixstyle else None
        if self.ms is not None and hasattr(self.bb, "stages"):
            for i in [0, 1, 2]:   # early stages only
                self.bb.stages[i].register_forward_hook(lambda mod, inp, out: self.ms(out))
    def forward(self, x):
        f = self.bb.forward_features(x)        # (B,C,h,w)
        if f.ndim == 4 and f.shape[1] != self.bb.num_features and f.shape[-1] == self.bb.num_features:
            f = f.permute(0, 3, 1, 2).contiguous()   # NHWC -> NCHW safety
        z = self.drop(self.pool(f))
        return self.cls(z), self.reg(z).squeeze(-1)

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            s = self.shadow[k]
            if v.dtype.is_floating_point: s.mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else: s.copy_(v)
    def copy_to(self, model): model.load_state_dict(self.shadow, strict=True)

# ----------------------------- data -----------------------------
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class ChemDataset(Dataset):
    def __init__(self, df, cfg, train, soft=None, cache_store=None):
        self.orig_idx = df.index.to_numpy()           # GLOBAL positions for OOF alignment
        self.df = df.reset_index(drop=True); self.cfg = cfg; self.train = train
        self.soft = soft
        self.y = self.df.interface_burden.values.astype(np.float32) if "interface_burden" in df else None
        self.load_h = int(cfg.img_h * 1.12)           # same scale for train & val (center-crop)
        self.load_w = int(cfg.img_w * 1.12)
        self.cache = cache_store    # dict idx->uint8 array, optional
        self.store = _STORE         # RAM store {id: (Hs,Ws,3)} if preprocessed
        self.ids = self.df.id.values
    def __len__(self): return len(self.df)
    def _load(self, i):
        if self.store is not None:                    # fast path: from RAM, no disk
            a = self.store[self.ids[i]]
            if a.shape[0] != self.load_h or a.shape[1] != self.load_w:
                a = cv2.resize(a, (self.load_w, self.load_h), interpolation=cv2.INTER_AREA)
            return a
        key = int(self.orig_idx[i])     # GLOBAL key: cache shared safely across folds
        if self.cache is not None and key in self.cache:
            return self.cache[key]
        p = Path(self.cfg.data_root) / self.df.image_path.iloc[i]
        im = Image.open(p).convert("RGB").resize((self.load_w, self.load_h), Image.BILINEAR)
        a = np.asarray(im, dtype=np.uint8)
        if self.cache is not None: self.cache[key] = a
        return a
    def __getitem__(self, i):
        import torchvision.transforms.v2.functional as TF
        a = self._load(i)
        x = torch.from_numpy(a).permute(2, 0, 1)          # (3,H,W) uint8
        H, W = self.cfg.img_h, self.cfg.img_w
        if self.train:
            # mild random crop (translation), horizontal flip ONLY
            top = random.randint(0, x.shape[1] - H); left = random.randint(0, x.shape[2] - W)
            x = x[:, top:top + H, left:left + W]
            if random.random() < 0.5: x = torch.flip(x, [2])
            x = x.float() / 255.0
            cj = self.cfg.color_jitter
            if cj > 0:
                x = TF.adjust_brightness(x, 1 + random.uniform(-cj, cj))
                x = TF.adjust_contrast(x, 1 + random.uniform(-cj, cj))
                x = TF.adjust_saturation(x, 1 + random.uniform(-cj, cj))
                x = TF.adjust_hue(x, random.uniform(-cj * 0.15, cj * 0.15))
            if random.random() < self.cfg.gray_p:
                g = x.mean(0, keepdim=True); x = g.repeat(3, 1, 1)
            if random.random() < 0.2:
                x = TF.gaussian_blur(x, kernel_size=3)
            if random.random() < 0.2:
                x = (x + torch.randn_like(x) * 0.02).clamp(0, 1)
        else:
            th = (x.shape[1] - H) // 2; lw = (x.shape[2] - W) // 2
            x = x[:, th:th + H, lw:lw + W].float() / 255.0
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        out = {"x": x, "idx": int(self.orig_idx[i])}
        if self.y is not None: out["y"] = self.y[i]
        if self.soft is not None: out["soft"] = torch.from_numpy(self.soft[i])
        return out

def cmixup(batch_x, soft, yreg, alpha):
    """C-Mixup: mix each sample with its nearest-in-burden neighbor in the batch."""
    order = torch.argsort(yreg)
    partner = torch.empty_like(order); partner[order] = torch.roll(order, 1)
    lam = float(np.random.beta(alpha, alpha))
    lam = max(lam, 1 - lam)
    x = lam * batch_x + (1 - lam) * batch_x[partner]
    s = lam * soft + (1 - lam) * soft[partner]
    yr = lam * yreg + (1 - lam) * yreg[partner]
    return x, s, yr

# ----------------------------- train one fold -----------------------------
def soft_ce(logits, target):                    # both (B,K)
    return -(target * F.log_softmax(logits, 1)).sum(1).mean()

def train_fold(cfg, df, fold_arr, fold, centers, device, cache=None, log=print):
    tr = df[fold_arr != fold]; va = df[fold_arr == fold]
    soft_tr = soft_ordinal_targets(tr.interface_burden.values, centers, cfg.sord_sigma)
    ds_tr = ChemDataset(tr, cfg, True, soft_tr, cache)
    ds_va = ChemDataset(va, cfg, False, None, cache)
    pw = cfg.num_workers > 0
    dl_tr = DataLoader(ds_tr, batch_size=cfg.batch_size, shuffle=True, drop_last=True,
                       num_workers=cfg.num_workers, pin_memory=True, persistent_workers=pw,
                       prefetch_factor=4 if pw else None)
    dl_va = DataLoader(ds_va, batch_size=cfg.batch_size * 2, shuffle=False,
                       num_workers=cfg.num_workers, pin_memory=True, persistent_workers=pw)
    model = Net(cfg.backbone, cfg.n_bins, cfg.pretrained, cfg.mixstyle, cfg.mixstyle_p, cfg.drop_path).to(device)
    head_ids = {id(p) for n, p in model.named_parameters() if n.startswith(("cls", "reg", "pool"))}
    params = [
        {"params": [p for p in model.parameters() if id(p) not in head_ids], "lr": cfg.lr},
        {"params": [p for p in model.parameters() if id(p) in head_ids], "lr": cfg.lr * cfg.head_lr_mult},
    ]
    opt = torch.optim.AdamW(params, weight_decay=cfg.weight_decay)
    steps = len(dl_tr) * cfg.epochs; warm = int(steps * cfg.warmup_frac)
    def lr_at(s):
        if s < warm: return s / max(1, warm)
        t = (s - warm) / max(1, steps - warm); return 0.5 * (1 + math.cos(math.pi * t))
    scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")
    ema = EMA(model, cfg.ema_decay)
    gstep = 0
    for ep in range(cfg.epochs):
        model.train(); t0 = time.time(); running = 0.0
        for b in dl_tr:
            x = b["x"].to(device, non_blocking=True); soft = b["soft"].to(device); yreg = b["y"].to(device)
            if cfg.cmix_p > 0 and random.random() < cfg.cmix_p:
                x, soft, yreg = cmixup(x, soft, yreg, cfg.cmix_alpha)
            for g in opt.param_groups: g["lr"] = (cfg.lr if g is opt.param_groups[0] else cfg.lr * cfg.head_lr_mult) * lr_at(gstep)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                cl, rg = model(x)
                loss = soft_ce(cl, soft) + cfg.reg_weight * F.binary_cross_entropy_with_logits(rg, yreg / 100.0)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            ema.update(model); gstep += 1; running += loss.item()
        log(f"  fold{fold} ep{ep+1}/{cfg.epochs} loss={running/len(dl_tr):.4f} {time.time()-t0:.0f}s")
    # OOF predict with EMA weights
    eval_model = Net(cfg.backbone, cfg.n_bins, False).to(device); eval_model.load_state_dict(ema.shadow); eval_model.eval()
    logits_all, reg_all, idx_all = [], [], []
    with torch.no_grad():
        for b in dl_va:
            x = b["x"].to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                cl, rg = eval_model(x)
            logits_all.append(cl.float().cpu().numpy()); reg_all.append(torch.sigmoid(rg).float().cpu().numpy() * 100)
            idx_all.append(b["idx"].numpy())
    return (np.concatenate(idx_all), np.concatenate(logits_all), np.concatenate(reg_all),
            {k: v.cpu() for k, v in ema.shadow.items()})

# ----------------------------- CV orchestration -----------------------------
def run_cv(cfg, log=print):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    set_seed(cfg.seed)
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(Path(cfg.data_root) / "train.csv")
    if cfg.smoke:
        df = df.groupby(to_zone(df.interface_burden.values)).head(150).reset_index(drop=True)
        log(f"SMOKE subset: {len(df)} rows")
    global _STORE
    _STORE = load_store("train")
    if _STORE is not None: log(f"RAM store loaded: {len(_STORE)} train images")
    groups = compute_groups(df, cfg.data_root)
    fold_arr = make_folds(df, groups, cfg.n_folds, cfg.fold_seed)
    log(f"groups={groups.max()+1} folds={cfg.n_folds} | fold sizes={np.bincount(fold_arr).tolist()}")
    centers = cfg.centers
    cache = {} if cfg.cache else None
    oof_logits = np.zeros((len(df), cfg.n_bins), np.float32); oof_reg = np.zeros(len(df), np.float32)
    run_folds = [int(x) for x in cfg.folds_to_run.split(",") if x != ""] or list(range(cfg.n_folds))
    for f in run_folds:
        idx, logits, reg, sd = train_fold(cfg, df, fold_arr, f, centers, device, cache, log)
        oof_logits[idx] = logits; oof_reg[idx] = reg
        torch.save({"sd": sd, "cfg": asdict(cfg)}, out / f"model_f{f}.pt")
    y = df.interface_burden.values
    done = np.isin(fold_arr, run_folds)
    # decision: temperature-calibrate on OOF then expected-cost
    T, _ = fit_temperature(oof_logits[done], y[done], centers)
    from scipy.special import softmax
    pmf = softmax(oof_logits[done] / T, 1)
    pred = expected_cost_decision(pmf, centers)
    comp = evaluate_components(y[done], pred)
    # baselines for context
    reg_only = evaluate(y[done], np.clip(oof_reg[done], 0, 100))
    pmf_exp = pmf @ centers  # PMF expectation as a point estimate
    exp_only = evaluate(y[done], np.clip(pmf_exp, 0, 100))
    from scipy.stats import spearmanr
    sp_reg = spearmanr(oof_reg[done], y[done]).correlation
    sp_exp = spearmanr(pmf_exp, y[done]).correlation
    log(f"\n=== OOF (folds {run_folds}, n={done.sum()}) ===")
    log(f"  T*={T}  expected-cost score={comp['total']:.4f}  (zone_acc={comp['zone_accuracy']:.3f}, "
        f"zone={comp['w_zone']:.3f} abs={comp['w_absolute']:.3f} high={comp['w_high']:.3f} ext={comp['w_extreme']:.3f})")
    log(f"  reg-head-only score={reg_only:.4f} | pmf-expectation score={exp_only:.4f}  (context)")
    log(f"  spearman(reg,y)={sp_reg:.3f}  spearman(pmf_exp,y)={sp_exp:.3f}  (did it learn?)")
    tz, pz = to_zone(y[done]), to_zone(pred)
    cm = np.zeros((4, 4), int)
    for a, b in zip(tz, pz): cm[a, b] += 1
    log("  zone confusion (rows=true 0..3, cols=pred 0..3):")
    for r in range(4): log("    ", cm[r].tolist())
    np.savez(out / "oof.npz", logits=oof_logits, reg=oof_reg, fold=fold_arr, y=y,
             ids=df.id.values, done=done, centers=centers, T=T)
    json.dump({"score": comp["total"], "T": T, "zone_acc": comp["zone_accuracy"],
               "components": {k: float(v) for k, v in comp.items()}, "folds": run_folds},
              open(out / "cv_result.json", "w"), indent=2)
    log(f"saved {out/'oof.npz'} and cv_result.json")
    return comp["total"]

def apply_decision(pmf, reg, centers, dc):
    """Map per-sample PMF (+reg) to a final burden value using a decision config dc.
    Strategies:
      expected_cost : Bayes-optimal grid search over the exact metric (metric.py)
      pmf_exp       : PMF expectation, clipped
      reg           : regression head, clipped
      blend_thresh  : s = w*pmf_exp + (1-w)*reg; re-zone by tuned cuts; clip s into
                      the predicted zone bounds (decouples the zone decision from
                      calibration -> tunes the hard Z2/Z3 boundary directly)."""
    s_exp = pmf @ centers
    strat = dc.get("strategy", "blend_thresh")
    if strat == "expected_cost":
        return np.clip(expected_cost_decision(pmf, centers), 0, 100)
    if strat == "pmf_exp":
        return np.clip(s_exp, 0, 100)
    if strat == "reg":
        return np.clip(reg, 0, 100)
    # blend_thresh
    w = dc.get("w", 1.0); cuts = np.asarray(dc.get("cuts", [12.0, 35.0, 48.0]))
    m = dc.get("margin", 0.5)
    s = w * s_exp + (1 - w) * reg
    z = np.digitize(s, cuts)                       # 0..3
    lo = SEVERITY_BINS[z]; hi = SEVERITY_BINS[z + 1]
    return np.clip(np.clip(s, lo + m, hi - m), 0, 100)


def predict_test(cfg, log=print):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out = Path(cfg.out_dir)
    test = pd.read_csv(Path(cfg.data_root) / "test.csv")
    global _STORE
    _STORE = load_store("test")
    centers = cfg.centers
    models = sorted(out.glob("model_f*.pt"))
    assert models, "no fold models found"
    oof = np.load(out / "oof.npz")
    dc = json.load(open(out / "decision.json")) if (out / "decision.json").exists() else {"strategy": "expected_cost"}
    T = float(dc.get("T", oof["T"]))
    ds = ChemDataset(test, cfg, False, None, None)
    dl = DataLoader(ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    from scipy.special import softmax
    pmf_sum = np.zeros((len(test), cfg.n_bins), np.float64)
    reg_sum = np.zeros(len(test), np.float64); nviews = 0
    for mp in models:
        ckpt = torch.load(mp, map_location=device)
        model = Net(cfg.backbone, cfg.n_bins, False).to(device); model.load_state_dict(ckpt["sd"]); model.eval()
        with torch.no_grad():
            ptr = 0
            for b in dl:
                x = b["x"].to(device)
                for xx in (x, torch.flip(x, [3])):    # hflip TTA
                    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                        cl, rg = model(xx)
                    pmf_sum[ptr:ptr + len(x)] += softmax(cl.float().cpu().numpy() / T, 1)
                    reg_sum[ptr:ptr + len(x)] += torch.sigmoid(rg).float().cpu().numpy() * 100
                ptr += len(x)
        nviews += 2
    pmf = pmf_sum / pmf_sum.sum(1, keepdims=True)
    reg = reg_sum / nviews
    pred = np.clip(apply_decision(pmf, reg, centers, dc), 0, 100)
    log(f"decision: {dc}")
    sub = pd.DataFrame({"id": test.id, "interface_burden": pred})
    sub.to_csv(out / "submission.csv", index=False)
    log(f"wrote {out/'submission.csv'} rows={len(sub)} pred[min/mean/max]={pred.min():.1f}/{pred.mean():.1f}/{pred.max():.1f}")
    log(f"pred zone dist={np.bincount(to_zone(pred), minlength=4).tolist()}")
    return sub


In [ ]:
_DECISION_OPT_SRC = '"""Tune the decision/post-processing on the full OOF (working/oof.npz), HONESTLY.\n\nThe metric is ~80% ordinal-zone, so the output->value map and zone cutpoints are a\nno-retrain lever. BUT threshold tuning overfits (esp. under high leave-experiment-out\nvariance). So we SELECT by the per-fold held-out (nested) score, not full-OOF, and\nconstrain cuts near the true bounds to avoid degenerate solutions. Robust default =\nexpected_cost (temperature-only). Saves working/decision.json for predict_test.\n"""\nimport os, sys, json\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.special import softmax\nfrom scipy.optimize import minimize\n\nOUT = Path(os.environ["OUT_DIR"])\noof = np.load(OUT / "oof.npz")\nlogits, reg_oof, y = oof["logits"], oof["reg"], oof["y"]\nfold, done, centers = oof["fold"], oof["done"], oof["centers"]\nm = done.copy()\nfolds = sorted(set(fold[m].tolist()))\nprint(f"OOF samples: {m.sum()} | folds: {folds}")\n\n# constrain cuts to windows around the true bounds [12,35,48] with a min gap\nLO = np.array([6.0, 28.0, 42.0]); HI = np.array([20.0, 41.0, 54.0])\ndef proj_cuts(c):\n    c = np.clip(np.asarray(c, float), LO, HI)\n    c[1] = max(c[1], c[0] + 4); c[2] = max(c[2], c[1] + 4)\n    return c\n\ndef score_dc(dc, mask):\n    pmf = softmax(logits[mask] / dc.get("T", 1.0), 1)\n    return evaluate(y[mask], apply_decision(pmf, reg_oof[mask], centers, dc))\n\ndef best_T(strategy, mask):\n    grid = list(np.round(np.r_[np.arange(0.4, 1.0, 0.1), np.arange(1.0, 3.51, 0.25)], 3))\n    return min(((float(T), score_dc({"strategy": strategy, "T": T}, mask)) for T in grid), key=lambda t: t[1])\n\ndef optimize_blend(mask):\n    best = (None, 1e9)\n    for w in [1.0, 0.7, 0.5, 0.3, 0.0]:\n        for init in ([12, 35, 48], [11, 33, 46], [13, 37, 50]):\n            def obj(c):\n                c = proj_cuts(c)\n                return score_dc({"strategy": "blend_thresh", "w": w, "cuts": c.tolist(), "T": 1.0, "margin": 0.5}, mask)\n            r = minimize(obj, np.array(init, float), method="Nelder-Mead",\n                         options={"xatol": 0.15, "fatol": 1e-4, "maxiter": 300})\n            if r.fun < best[1]:\n                best = ({"strategy": "blend_thresh", "w": w, "cuts": proj_cuts(r.x).round(3).tolist(),\n                         "T": 1.0, "margin": 0.5}, float(r.fun))\n    return best\n\nprint("\\n--- base strategies (full OOF, context only) ---")\nfor s in ["reg", "pmf_exp", "expected_cost"]:\n    T, sc = best_T(s, m); print(f"  {s:14s} bestT={T:<5} full-OOF={sc:.4f}")\ndc_blend_full, s_blend_full = optimize_blend(m)\nprint(f"  blend_thresh   full-OOF={s_blend_full:.4f}  (w={dc_blend_full[\'w\']} cuts={dc_blend_full[\'cuts\']})")\n\n# ---- HONEST per-fold held-out: tune on other folds, score the held-out fold ----\nprint("\\n--- per-fold held-out (HONEST selection) ---")\nec_oos, blend_oos = [], []\nfor f in folds:\n    tr = m & (fold != f); te = m & (fold == f)\n    Tt, _ = best_T("expected_cost", tr)\n    ec_oos.append(score_dc({"strategy": "expected_cost", "T": Tt}, te))\n    dcf, _ = optimize_blend(tr)\n    blend_oos.append(score_dc(dcf, te))\nec_h, blend_h = float(np.mean(ec_oos)), float(np.mean(blend_oos))\nprint(f"  expected_cost held-out: {ec_h:.4f} +/- {np.std(ec_oos):.3f}")\nprint(f"  blend_thresh  held-out: {blend_h:.4f} +/- {np.std(blend_oos):.3f}")\n\n# choose the strategy with the better HONEST held-out; fit its params on full OOF\nif blend_h + 0.15 < ec_h:           # require a clear margin to prefer the riskier tune\n    final = dc_blend_full\n    chosen_honest = blend_h\nelse:\n    T_ec, _ = best_T("expected_cost", m)\n    final = {"strategy": "expected_cost", "T": T_ec}\n    chosen_honest = ec_h\njson.dump(final, open(OUT / "decision.json", "w"), indent=2)\ncomp = evaluate_components(y[m], apply_decision(softmax(logits[m]/final.get("T",1.0),1), reg_oof[m], centers, final))\nprint(f"\\nSAVED decision.json: {final}")\nprint(f"  honest held-out estimate = {chosen_honest:.4f} | full-OOF = {comp[\'total\']:.4f} (zone_acc {comp[\'zone_accuracy\']:.3f})")\n'

In [ ]:
import os
os.environ.setdefault("DATA_ROOT", "./dataset/public")
os.environ.setdefault("OUT_DIR", "./working")
FAST = os.environ.get("FAST", "0") == "1"
DATA = os.environ["DATA_ROOT"]; OUT = os.environ["OUT_DIR"]
Path(OUT).mkdir(parents=True, exist_ok=True)

# config: light, well-tuned base configs (bigger/aggressive-aug overfit the experiments)
BASE = dict(N_FOLDS=5, IMG_H=320, IMG_W=192, EPOCHS=18, BATCH=32, NUM_WORKERS=8, FOLD_SEED=42)
CONFIGS = [("convnextv2_nano.fcmae_ft_in22k_in1k", 18), ("convnextv2_femto.fcmae_ft_in1k", 20)]
SEEDS = [42, 43, 44]
if FAST:
    CONFIGS = [("convnextv2_nano.fcmae_ft_in22k_in1k", 2)]; SEEDS = [42]; BASE["N_FOLDS"] = 3

run_dirs = []
for bb, ep in CONFIGS:
    for sd in SEEDS:
        d = f"{OUT}/{bb.split('.')[0]}_s{sd}"
        cfg = Config()                       # set fields directly (single-process notebook)
        cfg.data_root = DATA; cfg.out_dir = d; cfg.backbone = bb; cfg.epochs = ep; cfg.seed = sd
        cfg.fold_seed = BASE["FOLD_SEED"]; cfg.n_folds = BASE["N_FOLDS"]
        cfg.img_h = BASE["IMG_H"]; cfg.img_w = BASE["IMG_W"]; cfg.batch_size = BASE["BATCH"]
        cfg.num_workers = BASE["NUM_WORKERS"]
        print(f"=== {bb} seed {sd} ===")
        run_cv(cfg)
        run_dirs.append(d)
os.environ["OUT_DIR"] = OUT

# ensemble OOF -> honest decision -> ensemble test prediction
import numpy as np, pandas as pd, glob, torch
from scipy.special import softmax
oofs = [np.load(Path(d)/"oof.npz") for d in run_dirs]
pmf = np.mean([softmax(o["logits"],1) for o in oofs],0); reg = np.mean([o["reg"] for o in oofs],0)
o0 = oofs[0]; ens = Path(OUT)
np.savez(ens/"oof.npz", logits=np.log(pmf+1e-12).astype(np.float32), reg=reg.astype(np.float32),
         fold=o0["fold"], y=o0["y"], done=o0["done"], centers=o0["centers"], T=np.float64(1.0), ids=o0["ids"])
# honest decision (reuse decision logic)
exec(_DECISION_OPT_SRC)

# ensemble test predict (mixed backbones via each ckpt's cfg)
import json as _json
dc = _json.load(open(ens/"decision.json")); T = float(dc.get("T",1.0)); centers = o0["centers"]
test = pd.read_csv(Path(DATA)/"test.csv"); _STORE = None
mps = [m for d in run_dirs for m in sorted(glob.glob(f"{d}/model_f*.pt"))]
dev = "cuda" if torch.cuda.is_available() else "cpu"
ps = np.zeros((len(test),len(centers))); rs = np.zeros(len(test)); nv = 0
for mp in mps:
    ck = torch.load(mp, map_location=dev); c = ck["cfg"]
    cf = Config(); cf.backbone=c["backbone"]; cf.img_h=c["img_h"]; cf.img_w=c["img_w"]; cf.n_bins=c["n_bins"]; cf.data_root=DATA
    md = Net(cf.backbone, cf.n_bins, False).to(dev); md.load_state_dict(ck["sd"]); md.eval()
    dl = DataLoader(ChemDataset(test, cf, False, None, None), batch_size=64, shuffle=False, num_workers=8)
    with torch.no_grad():
        ptr=0
        for b in dl:
            x=b["x"].to(dev)
            for xx in (x, torch.flip(x,[3])):
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev=="cuda"):
                    cl,rg = md(xx)
                ps[ptr:ptr+len(x)] += softmax(cl.float().cpu().numpy()/T,1)
                rs[ptr:ptr+len(x)] += torch.sigmoid(rg).float().cpu().numpy()*100
            ptr+=len(x)
    nv+=2
pmf_t = ps/ps.sum(1,keepdims=True); reg_t = rs/nv
pred = np.clip(apply_decision(pmf_t, reg_t, centers, dc), 0, 100)
pd.DataFrame({"id":test.id,"interface_burden":pred}).to_csv(Path(OUT)/"submission.csv", index=False)
print("wrote", Path(OUT)/"submission.csv", "| pred zones", np.bincount(to_zone(pred),minlength=4).tolist())


In [ ]:
# strict validation
import pandas as pd, numpy as np
sub = pd.read_csv(Path(OUT)/"submission.csv"); samp = pd.read_csv(Path(DATA)/"sample_submission.csv")
assert list(sub.columns)==["id","interface_burden"]; assert len(sub)==len(samp)
assert sub.id.is_unique and set(sub.id)==set(samp.id)
v=sub.interface_burden.values; assert np.isfinite(v).all() and (v>=0).all() and (v<=100).all()
print("submission valid:", sub.shape)